# Ordered Chaos — Python Runtime Validation: Exploration Notebook

In [ ]:
import sys
import os
import csv
import math
import random
import statistics
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.unstable_object import UnstableObject
from core.descriptor_protocol import UnstableDescriptor, DescriptorHost
from analysis.sle_fit import fit_sle, build_sle_result

random.seed(42)
print('Imports OK')

In [ ]:
obj = UnstableObject(base=10.0)

print('Repeated read() on same UnstableObject:')
for i in range(6):
    val = obj.read()
    print(f'  read {i}: {val:.4f}  (access_count={obj.access_count-1}, entropy={obj.entropy-0.1:.2f})')

print()
print('observe() injects entropy permanently:')
snap = obj.observe()
print(f'  snapshot before inject : {snap}')
print(f'  entropy after observe  : {obj.entropy:.4f}')
val_after = obj.read()
print(f'  next read() after observe: {val_after:.4f}')

print()
print('DescriptorHost attribute access:')
DescriptorHost.x.reset()
host = DescriptorHost()
for i in range(4):
    v = host.x
    print(f'  host.x [{i}]: {v:.4f}')

print()
print('Shuffled vs fixed order divergence demo:')

def run_once(ops):
    o = UnstableObject(base=10.0)
    y = 0.0
    for op in ops:
        if op == 'add':
            y += o.read()
        elif op == 'observe':
            o.observe()
    return y * o.read()

base_ops = ['add'] * 6 + ['observe'] * 1
results = []
for _ in range(10):
    ops = base_ops[:]
    random.shuffle(ops)
    results.append(run_once(ops))

print(f'  10 shuffles -> min={min(results):.2f}  max={max(results):.2f}  range={max(results)-min(results):.2f}  std={statistics.stdev(results):.4f}')

In [ ]:
import functools

print('Manual dict cache invalidation:')
obj = UnstableObject(base=10.0)
first_val = obj.read() * 2.0
cache = {'key': first_val}
ac_snap = obj.access_count
ent_snap = obj.entropy

for steps in [0, 1, 3, 5, 10]:
    obj2 = UnstableObject(base=10.0, initial_entropy=ent_snap)
    obj2.access_count = ac_snap
    for _ in range(steps):
        obj2.read()
    true_val = obj2.read() * 2.0
    error_pct = abs(cache['key'] - true_val) / abs(true_val) * 100 if true_val != 0 else 0
    print(f'  steps_between={steps:>2}: cached={cache["key"]:.4f}  true={true_val:.4f}  error={error_pct:.2f}%')

print()
print('lru_cache invalidation (observe between calls):')
shared = UnstableObject(base=10.0)

@functools.lru_cache(maxsize=128)
def cached_read(key: str) -> float:
    return shared.read()

r1 = cached_read('x')
ac_after = shared.access_count
ent_after = shared.entropy
shared.observe()
r2_cached = cached_read('x')

fresh = UnstableObject(base=10.0, initial_entropy=shared.entropy)
fresh.access_count = ac_after
r2_true = fresh.read()

print(f'  first call      : {r1:.4f}')
print(f'  cache hit       : {r2_cached:.4f}')
print(f'  true next read  : {r2_true:.4f}')
print(f'  error           : {abs(r2_cached - r2_true):.4f} ({abs(r2_cached - r2_true)/abs(r2_true)*100:.2f}%)')
cached_read.cache_clear()

In [ ]:
def run_nonlin(nonlinearity, n=5000):
    results = []
    for _ in range(n):
        ops = ['add'] * 6 + ['observe'] * 1
        random.shuffle(ops)
        o = UnstableObject(base=10.0)
        y = 0.0
        for op in ops:
            if op == 'add':
                y += o.read()
            else:
                o.observe()
        if nonlinearity == 'linear':
            results.append(y)
        elif nonlinearity == 'quadratic':
            results.append(y * o.read())
        elif nonlinearity == 'cubic':
            results.append(y * o.read() * o.read())
        elif nonlinearity == 'extreme':
            results.append(y * y * o.read())
    return results

random.seed(42)
levels = ['linear', 'quadratic', 'cubic', 'extreme']
degree_map = {'linear': 1, 'quadratic': 2, 'cubic': 3, 'extreme': 4}

degrees, log_ranges, ranges = [], [], []
print(f'  {"Level":<12}  {"Degree":>6}  {"Range":>12}  {"log(Range)":>10}')
print('  ' + '-' * 46)
for level in levels:
    vals = run_nonlin(level, n=5000)
    r = max(vals) - min(vals)
    lr = math.log(r) if r > 1.0 else 0.0
    d = degree_map[level]
    print(f'  {level:<12}  {d:>6}  {r:>12.4f}  {lr:>10.4f}')
    if r > 1.0:
        degrees.append(d)
        log_ranges.append(lr)
        ranges.append(r)

sle, r2 = fit_sle(degrees, log_ranges)
print()
print(f'  SLE (Python substrate, n=5000): {sle:.4f}')
print(f'  R²                            : {r2:.4f}')
print(f'  (Hiesenoether reference SLE   : 2.7891)')

In [ ]:
SUMMARY_DIR = Path('../results/summary')

def load_csv(path):
    if not path.exists():
        print(f'  [missing] {path}')
        return []
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

files = [
    'sle_python_substrate.csv',
    'descriptor_experiments.csv',
    'cache_invalidation_summary.csv',
]

for fname in files:
    rows = load_csv(SUMMARY_DIR / fname)
    print(f'{fname}: {len(rows)} rows')
    if rows:
        keys = list(rows[0].keys())
        print(f'  columns: {keys}')
        for row in rows[:3]:
            print(f'  {dict(list(row.items())[:6])}')
    print()

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

FIGURES_DIR = Path('../results/figures')

figure_files = [
    'python_a1_observation.png',
    'python_sle_fit.png',
    'python_cache_invalidation.png',
    'python_a3_length_scaling.png',
]

for fname in figure_files:
    fpath = FIGURES_DIR / fname
    if fpath.exists():
        print(f'{fname}')
        display(Image(filename=str(fpath)))
    else:
        print(f'[missing] {fname} — run run_validation.py first')